In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Predicting House Prices on Kaggle

Fashion-MNIST arrives clean: every example is an image of the same size,
every pixel lies between 0 and 1, and every label comes from the same ten
classes. Most data that people want to predict from looks different. In this
section we predict the sale prices of houses in Ames, Iowa, from a table in
which each house is described by 79 attributes. Some are numbers, such as
the living area; others are categories, such as the type of roof; many
entries are missing; and the numbers range from 0 or 1 to tens of thousands.
Only 1,460 houses come with a price. A model cannot read such a table
directly, and with so few examples, a careless estimate of its error can be
off by more than the differences between the models we want to compare.

The task brings together much of what the book has covered so far: the squared
loss and minibatch SGD of that section, cross-validation
(that section) and weight decay
(that section), and the MLP of this chapter with the
initialization of that section. Most of the work lies in
preparing the data and in estimating the error; the models take a few lines
each.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import pandas as pd

## The Competition

### Kaggle

[Kaggle](https://www.kaggle.com) hosts machine learning competitions. Each
competition provides a training set with labels and a test set without
them. Participants upload predictions for the test set, and Kaggle scores
them against labels that it keeps hidden, so no participant can tune a model
on the test labels. The scores of all participants appear on a leaderboard.
The data of this section belong to the competition
[House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques),
a practice competition without a deadline. The data were collected by
@De-Cock.2011 from the sales of 2006 to 2010, as a larger and more
varied replacement for small teaching datasets such as Boston Housing.

### The Data

We download the training and the test table with `d2l.download` and read them
with `pandas` (that section).

In [ ]:
def read_house_prices(root='../data'):
    """Download and read the training and test tables of the competition."""
    train = pd.read_csv(d2l.download(
        d2l.DATA_URL + 'kaggle_house_pred_train.csv', root,
        sha1_hash='585e9cc93e70b39160e7921475f9bcd7d31219ce'))
    test = pd.read_csv(d2l.download(
        d2l.DATA_URL + 'kaggle_house_pred_test.csv', root,
        sha1_hash='fa19780a7b011d9b009e8bff8e99922a8ee2eb90'))
    return train, test

The training table has one row for each of 1,460 houses: an identifier, 79
attributes, and the sale price, the label. The test table has 1,459 houses,
with identifiers and attributes but no prices.

In [ ]:
raw_train, raw_test = read_house_prices()
print(raw_train.shape, raw_test.shape)

Here are the first four houses, with the first three attributes, the last
two, and the price:

In [ ]:
print(raw_train.iloc[:4, [0, 1, 2, 3, -3, -2, -1]])

`MSSubClass` encodes the type of dwelling as a number, `MSZoning` the zoning
of the lot as a category, and `LotFrontage` the length of the lot's street
front in feet; `SaleType` and `SaleCondition` describe the sale. Of the 79
attributes, 36 are stored as numbers and 43 as categories, and 19 have missing
entries. Most of these mean that a feature is absent: `PoolQC`, the quality of
the pool, is missing for 1,453 houses, which simply have no pool. The first
column, `Id`, numbers the rows. The competition defines it as an identifier,
so we do not use it as a feature.

### The Error Measure

Prices span a wide range: the cheapest house in the training table sold for
\$34,900, the most expensive for \$755,000. An error of \$50,000 is more than
the first house is worth and less than 7% of the second, so what matters is
the relative error, not the error in dollars. The competition therefore
measures the error on a logarithmic scale. If a prediction $\hat{y}$ satisfies
$|\log \hat{y} - \log y| \leq \delta$, then
$e^{-\delta} \leq \hat{y}/y \leq e^{\delta}$, and for small $\delta$ the
prediction lies within a fraction of about $\delta$ of the true price $y$.
Over the $n$ houses of the test set, the score is the root-mean-squared
error (RMSE) of the logarithms,

$$\sqrt{\frac{1}{n}\sum_{i=1}^n\left(\log y_i - \log \hat{y}_i\right)^2}.$$

An RMSE of 0.1 thus corresponds to typical errors of about 10% of the price.
We train our models to predict $\log y$ with the squared loss, so the
training loss is the square of the score computed on the training houses.

## Preprocessing

### Numerical Features

The numerical attributes have very different scales: the number of fireplaces
is a small integer, the area of the lot runs to tens of thousands of square
feet. A weight that suits one of them is far too large or too small for
another, which slows minibatch SGD. Weight decay adds a second reason. It
penalizes all weights alike, and so it would fall hardest on the small-valued
features, which need the largest coefficients. We therefore standardize each
numerical feature as in that section: with $\mu$ and $s$ the mean and
the standard deviation of the feature over the training rows, we replace its
value $x$ by $(x - \mu)/s$.

A missing numerical value is then replaced by 0, the mean of the standardized
feature. This is the mean imputation of that section, and it assumes
that a missing value is typical. For `LotFrontage`, missing for 259 houses,
this is a fair guess, although the variance of the filled column falls below
1, to about 0.82. For the year in which a garage was built it is a poor one,
since there a gap means that the house has no garage. Exercise
1 examines the bias that mean
imputation introduces in such a case.

### Categorical Features

A category cannot be multiplied by a weight. We therefore one-hot encode each
categorical feature with `get_dummies`, as in that section. `MSZoning`
takes five values in the training table, among them `RL` (residential, low
density) and `RM` (residential, medium density), so it becomes five indicator
features `MSZoning_RL`, `MSZoning_RM`, and so on. A missing value counts as
one more category, so that "no pool" gets its own column.

A column stored as numbers is not always a quantity. `MSSubClass` codes the
type of dwelling with numbers from 20 to 190: 20 stands for a one-story
house built in 1946 or later, 60 for a two-story house of the same period,
and 90 for a duplex. The differences between these codes measure nothing,
but with one weight on the standardized column, the prediction of a linear
model would change in proportion to them. We therefore convert the codes to
strings before the encoding, and `MSSubClass` becomes a categorical feature
with 15 indicators for the codes in the training table and one for a
missing value. The model thus sees 35 numerical and 44 categorical
attributes.

### Fitting on the Training Rows

Each preprocessing step uses statistics of the data: the means and standard
deviations of the numerical features, and the set of categories that become
columns. that section computed such statistics from the whole table
and warned that the shortcut should not be copied. Here we keep the promise.
The statistics come from the training rows alone, and the same transformation
is applied to all other rows, so that nothing about the rows on which we
measure the error enters the model before it is evaluated. The error estimate
would otherwise be too optimistic, a defect called *leakage*. The rule has a
consequence for the encoding: a category that does not occur in the training
rows gets no column, so a house with such a value has all of that feature's
indicators equal to 0. The `MSSubClass` code 150, for example, occurs in the
test table but not in the training table.

The function `preprocess` fits the transformation on the table `train` and
applies it to `train` and to any further tables. It returns NumPy arrays of
features, without the identifier and the price.

In [ ]:
def preprocess(train, *others):
    """Fit the feature preprocessing on train; apply it to all tables."""
    frames = [df.drop(columns=['Id', 'SalePrice'], errors='ignore')
                .astype({'MSSubClass': str})  # A code, not a quantity
              for df in (train, *others)]
    numeric = frames[0].select_dtypes(include='number').columns
    mean, std = frames[0][numeric].mean(), frames[0][numeric].std()
    for df in frames:
        df[numeric] = ((df[numeric] - mean) / std).fillna(0)
    frames = [pd.get_dummies(df, dummy_na=True) for df in frames]
    return [df.reindex(columns=frames[0].columns, fill_value=0)
              .to_numpy(dtype=np.float32) for df in frames]

Fitted on the whole training table, the encoding turns the 79 attributes
into 345 features:

In [ ]:
X_train, X_test = preprocess(raw_train, raw_test)
X_train.shape, X_test.shape

The label needs one step of preparation as well. The logarithms of the prices
lie around 12, with a standard deviation of 0.4. A model whose initial
predictions are near 0 would spend its first updates learning this offset,
which tells us nothing about any particular house (exercise
3 measures what this costs). We therefore
subtract the mean log price of the training rows from the targets, and add it
back to the predictions. Shifting targets and predictions by the same constant
leaves their differences, and hence the RMSE, unchanged.

The data module `KaggleHouse` holds one split of the labeled houses into
training and validation rows. It fits the preprocessing and the mean log
price on the training rows, and it serves the features together with the
centered log prices. A table without prices, such as the test table, can
take the place of the validation rows: the module then provides its
features for prediction but no validation data.

In [ ]:
class KaggleHouse(d2l.DataModule):
    """Features and centered log prices, fitted on the training rows."""
    def __init__(self, raw_train, raw_val, batch_size=64):
        super().__init__()
        self.save_hyperparameters(ignore=['raw_train', 'raw_val'])
        self.X_train, self.X_val = preprocess(raw_train, raw_val)
        self.num_inputs = self.X_train.shape[1]
        log_price = lambda df: np.log(
            df['SalePrice'].to_numpy(np.float32)).reshape(-1, 1)
        self.offset = log_price(raw_train).mean()  # Mean log price
        self.y_train = log_price(raw_train) - self.offset
        self.y_val = (log_price(raw_val) - self.offset
                      if 'SalePrice' in raw_val else None)

    def get_dataloader(self, train):
        X, y = ((self.X_train, self.y_train) if train
                else (self.X_val, self.y_val))
        if y is None:  # No prices to validate against
            return None
        return self.get_tensorloader((d2l.tensor(X), d2l.tensor(y)), train)

## Model Selection

### $K$-Fold Cross-Validation

To choose between models we need an estimate of their error on new houses. A
single validation set would be small: if we held out a fifth of the 1,460
houses, the estimate would rest on 292 of them, and it would depend on which
ones. We therefore use the $K$-fold cross-validation of
that section, in which every house serves for
validation once and the average of the $K$ validation errors is our estimate.
(In this section $K$ counts folds, not layers.) We use $K = 5$.

`k_fold_data` draws the folds and builds one `KaggleHouse` per round, so that
the preprocessing of each round is fitted on its own training folds. Fitting
it once on all labeled houses would be simpler, and it would leak the
statistics of each validation fold into its features.

In [ ]:
def k_fold_data(raw, k):
    """Split the table raw into k random folds; one data module per round."""
    folds = np.array_split(np.random.default_rng().permutation(len(raw)), k)
    return [KaggleHouse(raw.drop(index=raw.index[f]), raw.iloc[f])
            for f in folds]

`k_fold` trains a fresh model in each round and prints the training and
validation RMSE averaged over the rounds, together with the smallest and the
largest validation RMSE of a single fold. It returns the training and the
validation RMSE of every fold. Each of them is computed from the mean
squared error of log prices, averaged over the last five epochs in the
trainer's history; its square root is the RMSE of the competition. We
average for the
reason we averaged accuracies in that section: a single
epoch can land far off. This table gives minibatch SGD a particular
opportunity. Every house has 45 features equal to 1, the bias and one
indicator per categorical attribute, so the predictions as a whole move in
larger steps than any single weight, and now and then an epoch ends on such
a step.

The argument `make_model` builds a model for a given number of
input features. This number varies slightly from round to round, between about
335 and 345, because each round fits its own set of categories. Only the JAX
models need it: NNX layers take their number of inputs when they are
constructed, whereas the layers of the other frameworks infer it from the
first minibatch.

In [ ]:
def k_fold(folds, make_model, max_epochs=100):
    """Train a fresh model in each round; return the RMSEs of every fold."""
    rmse = []
    for fold in folds:
        trainer = d2l.Trainer(max_epochs=max_epochs)
        trainer.board.display = False
        trainer.fit(make_model(fold.num_inputs), fold)
        rmse.append([np.mean(trainer.history[f'{split}_loss'][-5:]) ** 0.5
                     for split in ('train', 'val')])  # Last five epochs
    train, val = np.array(rmse).T
    print(f'training RMSE {train.mean():.3f}, validation RMSE '
          f'{val.mean():.3f} (folds {val.min():.3f} to {val.max():.3f})')
    return train, val

We draw the folds once and evaluate every model on the same folds, so that
differences between the models do not come from different splits.

In [ ]:
folds = k_fold_data(raw_train, 5)

### A Linear Baseline

The first model is linear regression, `LinearRegression` of
that section, with weights drawn from
$\mathcal{N}(0, 0.01^2)$ and a zero bias. We train it, and every later
model, with minibatches of 64 houses at learning rate 0.02 for 100 epochs.
A baseline is only useful if it is trained as carefully as the models it is
compared with, so it gets the same budget.

In [ ]:
make_linear = lambda num_inputs: d2l.LinearRegression(lr=0.02)

In [ ]:
linear = k_fold(folds, make_linear)

The linear model predicts the validation folds with an RMSE of about 0.15, a
typical error of about 15% of the price, against about 0.11 on its training
folds. (The training figure is a running mean over the minibatches of the last
five epochs, not the result of a separate pass.) Single folds differ a lot: within
one run, the best fold has an RMSE of 0.11 to 0.13 and the worst one of about
0.2, depending on which houses they contain. The mean over the five folds
varies far less, by about a hundredth from one random split to the next. That
is still as much as the models differ. Luckily, models evaluated on the same
folds share much of this variation, so the difference between two of them is
measured more precisely than either error alone.

### An MLP with Weight Decay

Can an MLP do better? Ours has one hidden layer of 64 ReLU units and one
output, He-initialized weights (that section), and zero biases. It
reuses the squared loss of `LinearRegression`. With about 340 features it has
about 22,000 parameters, nineteen times as many as there are training houses
in a round, so we expect it to fit the training folds closely, including their
noise. Weight decay with strength `wd` (that section) limits
this; as there, we decay the weights but not the biases. Dropout
(that section) would be the other candidate, and one of the exercises
tries it.

In [ ]:
class HouseMLP(d2l.Module):
    """An MLP with one hidden layer, squared loss, and weight decay."""
    loss = d2l.LinearRegression.loss

    def __init__(self, num_hiddens, wd, lr):
        super().__init__()
        self.save_hyperparameters()
        # One initializer per layer: an instance repeats its random values
        he = lambda: tf.keras.initializers.VarianceScaling(
            2.0, 'fan_in', 'untruncated_normal')
        self.net = tf.keras.models.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu',
                                  kernel_initializer=he()),
            tf.keras.layers.Dense(1, kernel_initializer=he())])

    def configure_optimizers(self):
        optimizer = tf.keras.optimizers.SGD(self.lr, weight_decay=self.wd)
        optimizer.exclude_from_weight_decay(var_names=['bias'])
        return optimizer

make_mlp = lambda num_inputs, wd: HouseMLP(num_hiddens=64, wd=wd, lr=0.02)

We train the MLP with the learning rate, minibatch size, and number of
epochs of the linear model, on the same folds, for six strengths of weight
decay, from 0.001 to 0.3, each three times the one before, and plot the
average training and validation RMSE against the strength, with the
validation RMSE of the linear model for reference.

In [ ]:
wds = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
mlp = {}
for wd in wds:
    print(f'weight decay {wd}: ', end='')
    mlp[wd] = k_fold(folds, lambda num_inputs: make_mlp(num_inputs, wd))
d2l.plot(wds, [[mlp[wd][0].mean() for wd in wds],
               [mlp[wd][1].mean() for wd in wds], [linear[1].mean()] * 6],
         'weight decay', 'RMSE', xscale='log', fmts=('C0-o', 'C1-o', 'k:'),
         legend=['MLP, training', 'MLP, validation', 'linear, validation'],
         legend_loc='outside', figsize=(4.5, 3))

The validation curve is the U of that section, traced
here by the strength of weight decay instead of the degree of a polynomial.
At its left end, where a decay of 0.001 hardly restrains the weights, the
MLP fits the training folds more closely than the linear model did, with a
training RMSE of about 0.09 against 0.11, and predicts the validation folds
worse, at about 0.16, above the dotted line. It has fit noise. As the decay
grows, the training curve rises and the validation curve falls below the
dotted line, to its minimum of about 0.135 at a decay of 0.03. At the right
end the two curves all but meet, a little below 0.17: the weights are now
held so
small that the MLP no longer fits even the training folds as well as the
linear model does. At the minimum, the MLP is ahead of the linear model by
about a hundredth, which lowers the typical error from about 15% to 13% or
14% of the price. For all its 22,000 parameters, the MLP finds little in
this table that a weighted sum had missed.

The folds are shared, so we can also compare the best MLP with the linear
model fold by fold:

In [ ]:
best = min(wds, key=lambda wd: mlp[wd][1].mean())
print(f'weight decay {best}: MLP minus linear, per fold',
      np.round(mlp[best][1] - linear[1], 3))

The MLP is ahead on most folds, by up to about four hundredths, and on the
others the two models are within half a hundredth of each other. Its
advantage holds up fold by fold, and it is small.

## Submitting Predictions

Cross-validation has selected a model: the MLP with the weight decay of the
lowest validation RMSE. Each of its five models was trained on four fifths of
the houses. For the submission we train it once more, on all 1,460, with the
same hyperparameters, as in that section. The test
table takes the place of the validation rows, so the preprocessing and the
mean log price are fitted on the whole training table. The model predicts
centered log prices; we add the mean log price back, exponentiate, and write
one line per house to the file `submission.csv`.

In [ ]:
data = KaggleHouse(raw_train, raw_test)
model = make_mlp(data.num_inputs, wd=best)
trainer = d2l.Trainer(max_epochs=100)
trainer.board.display = False
trainer.fit(model, data)
log_price = d2l.numpy(model(d2l.tensor(data.X_val))).reshape(-1) + data.offset
submission = pd.DataFrame({'Id': raw_test['Id'],
                           'SalePrice': np.exp(log_price)})
submission.to_csv('submission.csv', index=False)
submission.head()

Each line of the file holds the identifier of a test house and its predicted
price in dollars, the format that the competition expects. To submit the file,
sign in to Kaggle, open the competition page, choose "Submit Prediction", and
upload `submission.csv`. Kaggle computes the score and shows it with the
position that the submission would take on the leaderboard. Do not expect the
score to match the cross-validated RMSE. The model trained on all houses
differs from the five models that were evaluated, and the weight decay was
selected among six candidates on the same folds that estimate its error,
which makes the estimate a little too favorable.

Retraining is not the only way to produce the predictions. The five models of
the selected configuration could instead each predict the test houses, and
their log-price predictions could be averaged; exercise
6 compares the two.

Nothing in this pipeline is specific to neural networks. The same
preprocessing, the same loss, and the same folds serve any model that predicts
a number from a vector of features. On tables like this one, ensembles of
decision trees trained by gradient boosting often predict better than MLPs
[@Grinsztajn.Oyallon.Varoquaux.2022; @Shwartz-Ziv.Armon.2022], and exercise
4 measures how much better on this dataset.

This is a sobering place for a chapter on MLPs to end, and an instructive one.
We asked in that section for a model in which the evidence from
one input depends on the others, and a hidden layer provides it. Of the three
problems that hidden layers raised, two are solved: backpropagation computes
the gradients, and the right initial scale lets them reach every layer. The
third, why a network that could memorize its data generalizes instead, we
could measure but not settle. A fully connected layer, however, knows nothing
of how its inputs are arranged. To it a house is a list of 345 numbers and an
image a list of 784. Neural networks have the advantage where the inputs have
structure that a layer can exploit, such as the neighboring pixels of an
image or the successive words of a sentence. The book turns there after one
chapter on the software that all of its models share.

## Summary

A table of mixed numerical and categorical attributes becomes a matrix of
features in three steps: standardize the numerical features, replace their
missing values by the mean, and encode the categories, including numbers that
only name categories, as indicator features, with one more indicator for a
missing value. All statistics of these steps come from the training rows only;
otherwise the error estimate leaks information from the rows it is computed
on. Prices matter relative to their size, so we predict their logarithms with
the squared loss, which trains the model on the square of the competition's
score. $K$-fold cross-validation estimates the error of each candidate from
all labeled houses, and evaluating every candidate on the same folds makes
their comparison fair. An MLP with 64 hidden units fits the noise of the
training houses unless weight decay restrains it. With the weight decay that
cross-validation selects, it predicts slightly better than a linear model.

## Exercises

1. [code] **Mean imputation.** `preprocess` replaces every missing
   numerical value by the training-set mean of its column. Its call to
   `pd.get_dummies` with `dummy_na=True` also adds a missing-value
   indicator column for every categorical feature.
    1. Identify a numerical feature in this dataset whose missingness is not
       random with respect to the sale price.
    1. Explain what bias mean imputation alone would introduce for that
       feature, and in which direction it would shift the predictions for
       the affected houses. Then determine whether the indicator columns
       that `preprocess` already creates let a linear model remove this
       bias.
    1. Replace mean imputation by median imputation, and separately add a
       binary missingness indicator column for each imputed feature. Compare
       the cross-validated errors of the three variants.

    *Adapted from Kaggle Learn,
    [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning),
    Exercise 2.*
1. [code] **Numbers that name categories.** `preprocess` encodes
   `MSSubClass` as a category because its codes name types of dwelling.
    1. Find the other numerical columns with at most a dozen distinct
       values, among them `MoSold`, `YrSold`, `OverallQual`, and
       `OverallCond`. For each, decide whether its values are quantities,
       ordered levels, or names, and whether one weight on the standardized
       value can express its effect on the log price. Which of them is
       cyclic, so that its largest and smallest values are neighbors?
    1. One-hot encode the columns that you judged to be categories, and
       compare the cross-validated RMSE of the linear model and of the
       selected MLP with that of this section on the same folds. Is the
       difference larger than the variation between random splits?
1. [code] **Scaling the features and the target.** Remove the
   standardization from `preprocess`. Because `fillna(0)` imputes the
   column mean only for standardized columns, fill missing numerical values
   with the training-set column means explicitly instead.
    1. Rerun the cross-validation of the linear model at the section's
       learning rate of $0.02$, and describe the training loss.
    1. Find, to within a factor of ten, the largest learning rate at which
       the unstandardized linear model trains without diverging. Compare
       its cross-validated error after $100$ epochs with that of the
       standardized pipeline, and explain both observations.
    1. Restore the standardization, but train on the log prices without
       subtracting their mean. Compare the cross-validated error after
       $100$ epochs with that of the centered targets, and explain the
       difference.
    1. With the section's pipeline, record the validation RMSE of the linear
       model and the mean of its residuals on the validation fold after
       every epoch. In the epochs where the RMSE jumps, what has moved?
       Every house has 45 features equal to 1. Explain why a step of SGD
       changes the mean prediction more than it changes any single weight,
       and check whether halving the learning rate removes the jumps.
1. [code] **Gradient-boosted trees.** On tabular data like this,
   ensembles of decision trees often predict better than neural networks
   [@Grinsztajn.Oyallon.Varoquaux.2022; @Shwartz-Ziv.Armon.2022]. Fit a
   gradient-boosted tree model (for example scikit-learn's
   `GradientBoostingRegressor`, or XGBoost or LightGBM if installed) to the
   centered log prices of the training rows of each fold in `folds`, and
   compute its validation RMSE on each fold. Compare the average with those
   of this section's models. Which properties of the data favor trees?
1. [code] **Target leakage.** A feature that is recorded only after, or
   because of, the event being predicted inflates validation scores without
   helping a deployed model.
    1. Identify at least one raw feature in this dataset that could
       plausibly be filled in only after the sale, and explain why
       cross-validation does not catch the problem.
    1. Verify the mechanism empirically: add a synthetic feature equal to
       the log sale price plus small Gaussian noise, rerun the
       cross-validation, and report the change in the validation RMSE.

    *Adapted from Kaggle Learn,
    [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning),
    Exercise 7.*
1. [code] **Fold ensembling versus retraining.**
   that section retrains the selected MLP on all
   houses. Alternatively, the five models trained during cross-validation
   could each predict the test houses, with the preprocessing of their own
   round, and their log-price predictions could be averaged.
    1. Implement this fold ensemble and submit both its predictions and
       those of the retrained model. Compare the two leaderboard scores with
       each other and with the cross-validated RMSE.
    1. Each model of the ensemble was trained on four fifths of the houses.
       Why might the ensemble nevertheless predict better than the model
       retrained on all of them?
    1. Explain why the training error of the retrained model is not a
       substitute for the cross-validated estimate.
1. [code] **Target encoding.** ● *Target encoding* replaces each category
   of a categorical column by the mean log sale price of the training rows
   in that category, so that the feature occupies a single column instead
   of the one column per category that `preprocess` creates.
    1. Target-encode every categorical column with more than ten categories
       instead of one-hot encoding it, computing the category means from
       the training rows of each fold. Give a category that does not occur
       in those rows the overall mean, and standardize the encoded columns
       like the numerical features. Compare the cross-validated error with
       that of one-hot encoding.
    1. The encoding above computes each training row's value from a mean
       that includes that row's own label. Encode the training rows out of
       fold instead: split the training rows of each fold into five parts,
       and encode each part with category means computed from the other four
       parts. Compare the cross-validated error with that of the in-sample
       encoding, and explain how the size of a category affects the
       difference between the two encodings.
1. [code] **Improving the score.** ● Lower the cross-validated RMSE of the
   MLP, and then its leaderboard score.
    1. Vary the width and the number of hidden layers, the learning rate,
       and the number of epochs, and tune the weight decay of each
       configuration on the same folds. Which change helps most?
    1. Replace weight decay by dropout (that section) after the
       hidden layer, and tune its probability. Does dropout help on this
       dataset, alone or together with weight decay?
    1. You have now compared many configurations on the same five folds.
       Why is the lowest cross-validated RMSE among them a biased estimate
       of the leaderboard score of the chosen configuration? Check how large
       the bias is by drawing new folds.
1. **Geometric averaging.** ● An ensemble of $K$ models that predict the log
   prices $z_1, \ldots, z_K$ of a house can average them before
   exponentiating, which amounts to a geometric mean in price space.
    1. Compare $\exp\bigl(\frac{1}{K}\sum_{k=1}^K z_k\bigr)$ with the
       price-space average $\frac{1}{K}\sum_{k=1}^K e^{z_k}$. Show which of
       the two never exceeds the other, and state when they coincide.
    1. Let $\log y$ be the true log price of the house. Show that the
       log-space average satisfies

       $$\Bigl(\log y - \frac{1}{K}\sum_{k=1}^K z_k\Bigr)^2 \leq \frac{1}{K}\sum_{k=1}^K \left(\log y - z_k\right)^2.$$

       Determine whether the logarithm of the price-space average satisfies
       the same bound for all predictions $z_1, \ldots, z_K$.

[Discussions](https://d2l.discourse.group/t/237)